In [ ]:
from collections import defaultdict
import numpy as np
import random
from tqdm import tqdm

In [18]:
cnot1 = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

swap = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]])

cnot2 = swap @ cnot1 @ swap

cnot2

array([[1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0],
       [0, 1, 0, 0]])

In [ ]:
qubit_dependancies = {
    1: (2, 3),
    2: (5, 6),
    3: (1, 2),
    4: (3, 4),
    5: (2, 3),
    6: (4, 6),
    7: (4, 5),
    8: (1, 4),
}

topology = {1: {2, 3}, 2: {1, 4}, 3: {1, 4, 5}, 4: {2, 3, 6}, 5: {3, 6}, 6: {4, 5}}


mapping = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}


DAG = {1: {3, 4}, 2: {6, 7}, 3: {5, 8}, 4: {5, 6}, 5: set(), 6: {7}, 7: {8}, 8: set()}


def get_front_layer(dag) -> set:
    frontend = set(dag.keys())
    for _, dependencies in dag.items():
        frontend = frontend - dependencies
    return frontend


def executable(gate, qubit_dependancies, topology, mapping):
    q1, q2 = qubit_dependancies[gate]
    Q1, Q2 = mapping[q1], mapping[q2]
    return Q2 in topology[Q1]


def remove_gate(gate, dag):
    new_dag = {}
    for g, dependencies in dag.items():
        if g == gate:
            continue
        new_dependencies = dependencies - {gate}
        new_dag[g] = new_dependencies
    return new_dag


def get_swap_candidates(mapping, topology):
    reverse_mapping = {v: k for k, v in mapping.items()}
    candidates = []
    for Q, neighbors in topology.items():
        for neighbor in neighbors:
            candidates.append((reverse_mapping[Q], reverse_mapping[neighbor]))

    return candidates


def get_distance_matrix(topology):
    n = len(topology)
    distance_matrix = np.full((n, n), np.inf)
    for i in range(n):
        distance_matrix[i][i] = 0
    for Q, neighbors in topology.items():
        for neighbor in neighbors:
            distance_matrix[Q - 1][neighbor - 1] = 1
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if (
                    distance_matrix[i][j]
                    > distance_matrix[i][k] + distance_matrix[k][j]
                ):
                    distance_matrix[i][j] = (
                        distance_matrix[i][k] + distance_matrix[k][j]
                    )
    return distance_matrix


def heuristic(F, dag, mapping, D, swap, decay, w=0.2, lookahead=2):
    h = 0
    E = set()
    lookahead_size = lookahead * len(F)

    for gate in F:
        E = E.union(dag[gate])
        if len(E) >= lookahead_size:
            break

    while len(E) < lookahead_size:
        new_E = set()
        for gate in E:
            new_E = new_E.union(dag[gate])
            if (len(E) + len(new_E)) >= lookahead_size:
                break
        if len(E.union(new_E)) == len(E):
            break

        E = E.union(new_E)

    for gate in F:
        q1, q2 = qubit_dependancies[gate]
        Q1, Q2 = mapping[q1], mapping[q2]
        h += D[Q1 - 1][Q2 - 1] / len(F)

    for gate in E:
        q1, q2 = qubit_dependancies[gate]
        Q1, Q2 = mapping[q1], mapping[q2]
        h += w * D[Q1 - 1][Q2 - 1] / len(E)

    h *= max(decay[swap[0]], decay[swap[1]])

    return h


def reverse_dag(dag):
    reversed_dag = {g: set() for g in dag.keys()}
    for g, dependencies in dag.items():
        for dep in dependencies:
            reversed_dag[dep].add(g)
    return reversed_dag


def make_random_circuit(n_gates: int, n_qubits: int):
    gate_dependancies = {}
    for i in range(1, n_gates + 1):
        q1 = random.randint(1, n_qubits)
        q2 = random.randint(1, n_qubits)
        while q2 == q1:
            q2 = random.randint(1, n_qubits)

        gate_dependancies[i] = (q1, q2)

    dag = {i: set() for i in range(1, n_gates + 1)}
    qubit_stacks = {i: [] for i in range(1, n_qubits + 1)}
    for gate, (q1, q2) in gate_dependancies.items():
        dep1 = qubit_stacks[q1][-1] if qubit_stacks[q1] else None
        dep2 = qubit_stacks[q2][-1] if qubit_stacks[q2] else None

        if dep1 is not None:
            dag[dep1].add(gate)
        if dep2 is not None:
            dag[dep2].add(gate)

        qubit_stacks[q1].append(gate)
        qubit_stacks[q2].append(gate)

    return dag, gate_dependancies

In [487]:
def sabre(n_qubits, dag, mapping, dist, w=0.5, delta=0.1, debug=False):
    front_layer = get_front_layer(dag)
    executions = []
    decay = {i: 1 for i in range(1, n_qubits + 1)}
    swaps = []

    while len(front_layer) > 0:
        execute_list = []
        for gate in front_layer:
            if executable(gate, qubit_dependancies, topology, mapping):
                execute_list.append(gate)
                executions.append(gate)

        if len(execute_list) > 0:
            for gate in execute_list:
                front_layer.remove(gate)
                dag = remove_gate(gate, dag)
            front_layer = get_front_layer(dag)
        else:
            score = {}
            swap_candidates = get_swap_candidates(mapping, topology)
            for swap in swap_candidates:
                new_mapping = mapping.copy()
                new_mapping[swap[0]], new_mapping[swap[1]] = (
                    new_mapping[swap[1]],
                    new_mapping[swap[0]],
                )
                score[swap] = heuristic(
                    front_layer, dag, new_mapping, dist, swap, decay, w=w
                )
            if debug:
                print(np.sort(np.array(list(score.values()), dtype=np.float32)))
                print("---------")
            best_swap = min(score, key=score.get)
            executions.append(f"swap {best_swap}")
            mapping[best_swap[0]], mapping[best_swap[1]] = (
                mapping[best_swap[1]],
                mapping[best_swap[0]],
            )
            decay[best_swap[0]] += delta
            decay[best_swap[1]] += delta
            swaps.append(best_swap)

    # calc swap depth
    affected_qubits = set()
    depth = 1 if len(swaps) > 0 else 0
    for swap in swaps:
        if swap[0] in affected_qubits or swap[1] in affected_qubits:
            depth += 1
            affected_qubits = set()

        affected_qubits.add(swap[0])
        affected_qubits.add(swap[1])

    count = len(swaps)

    return mapping, executions, count, depth


swaps = []
depths = []

n_qubits = 6
n_gates = 30
dist = get_distance_matrix(topology)

for _ in tqdm(range(1000)):
    n_qubits = 6
    mapping = {
        i + 1: v
        for i, v in enumerate(random.sample(list(range(1, n_qubits + 1)), k=n_qubits))
    }
    DAG, qubit_dependancies = make_random_circuit(n_gates, n_qubits)

    for _ in range(2):
        mapping, exec, count, depth = sabre(
            n_qubits, DAG.copy(), mapping.copy(), dist, w=0.7, delta=0.1
        )
        mapping, exec, count, depth = sabre(
            n_qubits, reverse_dag(DAG), mapping.copy(), dist, w=0.7, delta=0.1
        )

    mapping, exec, count, depth = sabre(
        n_qubits, DAG.copy(), mapping.copy(), dist, w=0.5, delta=0.1, debug=False
    )

    swaps.append(count)
    depths.append(depth)

print(np.mean(swaps), np.quantile(swaps, (0, 0.5, 1)))
print(np.mean(depths), np.quantile(depths, (0, 0.5, 1)))

100%|██████████| 1000/1000 [00:04<00:00, 226.38it/s]

10.409 [ 4. 10. 17.]
6.401 [ 2.  6. 12.]


In [ ]:
def monte_carlo_sabre(
    n_qubits,
    dag,
    mapping,
    dist,
    w=0.5,
    delta=0.1,
    k=5,
    depth=10,
    temperature=1.0,
    debug=False,
):
    front_layer = get_front_layer(dag)
    executions = []
    decay = {i: 1 for i in range(1, n_qubits + 1)}
    swaps = []

    while len(front_layer) > 0:
        execute_list = []
        for gate in front_layer:
            if executable(gate, qubit_dependancies, topology, mapping):
                execute_list.append(gate)
                executions.append(gate)

        if len(execute_list) > 0:
            for gate in execute_list:
                front_layer.remove(gate)
                dag = remove_gate(gate, dag)
            front_layer = get_front_layer(dag)
        else:
            swap_candidates = get_swap_candidates(mapping, topology)
            prior = []
            for swap in swap_candidates:
                new_mapping = mapping.copy()
                new_mapping[swap[0]], new_mapping[swap[1]] = (
                    new_mapping[swap[1]],
                    new_mapping[swap[0]],
                )
                prior.append(
                    heuristic(front_layer, dag, new_mapping, dist, swap, decay, w=w)
                )
            if debug:
                print(np.sort(np.array(prior, dtype=np.float32)))
                print("---------")

            prior = np.array(prior)
            probabilities = np.exp(-prior / temperature) / np.sum(
                np.exp(-prior / temperature)
            )
            # sample k candidates according to the probabilities
            sampled_swaps = np.random.choice(
                len(swap_candidates),
                size=min(k, len(swap_candidates)),
                replace=False,
                p=probabilities,
            )
            print(sampled_swaps)

            executions.append(f"swap {best_swap}")
            mapping[best_swap[0]], mapping[best_swap[1]] = (
                mapping[best_swap[1]],
                mapping[best_swap[0]],
            )
            decay[best_swap[0]] += delta
            decay[best_swap[1]] += delta
            swaps.append(best_swap)

    # calc swap depth
    affected_qubits = set()
    depth = 1 if len(swaps) > 0 else 0
    for swap in swaps:
        if swap[0] in affected_qubits or swap[1] in affected_qubits:
            depth += 1
            affected_qubits = set()

        affected_qubits.add(swap[0])
        affected_qubits.add(swap[1])

    count = len(swaps)

    return mapping, executions, count, depth

In [87]:
x = torch.tensor([[[[1, 2], [3, 4]], [[5, 6], [7, 8]], [[9, 10], [11, 12]]]])


print(x.shape)
print(x)
print(x.view(x.size(0), -1, x.size(-1)))
print(x.flatten(start_dim=1, end_dim=2))

torch.Size([1, 3, 2, 2])
tensor([[[[ 1,  2],
          [ 3,  4]],

         [[ 5,  6],
          [ 7,  8]],

         [[ 9, 10],
          [11, 12]]]])
tensor([[[ 1,  2],
         [ 3,  4],
         [ 5,  6],
         [ 7,  8],
         [ 9, 10],
         [11, 12]]])
tensor([[[ 1,  2],
         [ 3,  4],
         [ 5,  6],
         [ 7,  8],
         [ 9, 10],
         [11, 12]]])


In [161]:
import torch
from qiskit import QuantumCircuit

qs = QuantumCircuit(3)

qs.csx(0, 2)
qs.csx(1, 2)
qs.cx(0, 1)

qs.csx(1, 2)

qs.cx(0, 1)

print(qs)

                                 
q_0: ──■───────────■──────────■──
       │         ┌─┴─┐      ┌─┴─┐
q_1: ──┼─────■───┤ X ├──■───┤ X ├
     ┌─┴──┐┌─┴──┐└───┘┌─┴──┐└───┘
q_2: ┤ Sx ├┤ Sx ├─────┤ Sx ├─────
     └────┘└────┘     └────┘     


In [31]:
states = [3, 2, 3, 1, 3, 2, 1, 0]
pik = ["hej", "med", "dig", "hvad", "hedder", "du", "jeg", "hedder"]

mcts_policies = list(zip(states, pik))

root_state = mcts_policies[-1][0]

data = []
shortest_paths = defaultdict(lambda: float("inf"))
shortest_paths[root_state] = 0

prev_state = root_state
for state, pi in mcts_policies[::-1]:
    h = min(shortest_paths[state], shortest_paths[prev_state] + 1)
    shortest_paths[state] = h
    prev_state = state

    data.append((state, pi, h))

# visited = set()
# for (state, pi) in mcts_policies[::-1]:
#     if state not in visited:
#         data.append((state, pi, shortest_paths[state]))
#         visited.add(state)

print(data)

[(0, 'hedder', 0), (1, 'jeg', 1), (2, 'du', 2), (3, 'hedder', 3), (1, 'hvad', 1), (3, 'dig', 2), (2, 'med', 2), (3, 'hej', 2)]


In [23]:
import heapq

heap = []

heapq.heappush(heap, (3, "hej"))
heapq.heappush(heap, (3, "hej"))


[heapq.heappop(heap) for _ in range(2)]

[(3, 'hej'), (3, 'hej')]

In [27]:
import math


def g(x1, x2):
    return x1**2 + 2 * x2**2 - 2 * x1 * x2 - 2 * x2 + 2


def f(x1, x2):
    return math.log(g(x1, x2))


def gx1(x1, x2):
    return 2 * x1 - 2 * x2


def gx2(x1, x2):
    return 4 * x2 - 2 * x1 - 2


def grad_x1(x1, x2):
    return 1 / g(x1, x2) * gx1(x1, x2)


def grad_x2(x1, x2):
    return 1 / g(x1, x2) * gx2(x1, x2)


v1 = 0.0
v2 = 0.0
alpha = 0.9
lr = 0.5

x1 = -5.0
x2 = -5.0

for _ in range(100):
    v1 = alpha * v1 - lr * grad_x1(x1, x2)
    v2 = alpha * v2 - lr * grad_x2(x1, x2)

    x1 += v1
    x2 += v2

print((x1, x2))

(1.0225472103000495, 1.009118075991333)


In [25]:
from qiskit import QuantumCircuit

qs = QuantumCircuit(4)

qs.cx(0, 1)
qs.cx(2, 3)

print(qs)

qs

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘
q_2: ──■──
     ┌─┴─┐
q_3: ┤ X ├
     └───┘


In [ ]:
import numpy as np

kernel = np.array([[4, 0, 0], [0, 2, 3], [0, 3, 5]])

normalized_kernal = np.zeros_like(kernel, dtype=np.float32)

for i in range(kernel.shape[0]):
    for j in range(kernel.shape[1]):
        normalized_kernal[i][j] = kernel[i][j] / math.sqrt(kernel[i][i] * kernel[j][j])

normalized_kernal

array([[1.       , 0.       , 0.       ],
       [0.       , 1.       , 0.9486833],
       [0.       , 0.9486833, 1.       ]], dtype=float32)

In [150]:
min(None, 1)

TypeError: '<' not supported between instances of 'int' and 'NoneType'

In [514]:
k = 2
swap_candidates = [f"hej_{i}" for i in range(4)]
prior = np.random.rand(4)
temperature = 0.3
probabilities = np.exp(-prior / temperature) / np.sum(np.exp(-prior / temperature))
# sample k candidates according to the probabilities
sampled_swaps = np.random.choice(
    len(swap_candidates),
    size=min(k, len(swap_candidates)),
    replace=False,
    p=probabilities,
)

In [ ]:
from qiskit.circuit.quantumcircuit import QuantumCircuit


qs = QuantumCircuit(8)


qs.swap(0, 1)
qs.swap(2, 3)
qs.swap(2, 3)
qs.swap(4, 5)


print(qs)

           
q_0: ─X────
      │    
q_1: ─X────
           
q_2: ─X──X─
      │  │ 
q_3: ─X──X─
           
q_4: ─X────
      │    
q_5: ─X────
           
q_6: ──────
           
q_7: ──────
           


AttributeError: 'NoneType' object has no attribute 'final_index_layout'

In [ ]:
x = torch.zeros(4, 4, 5, dtype=torch.float32)
x[0][3][0] = 1.0
x[2][1][0] = 1.0
x[2][2][0] = 1.0

lefts, rights = torch.where(x[:, :, 0] == 1.0)
pairs = list(zip(lefts.tolist(), rights.tolist()))

pairs

[(0, 3), (2, 1), (2, 2)]

In [ ]:
DAGCircuit.add_captured_stretch

In [520]:
sampled_swaps = np.random.choice(
    len(swap_candidates),
    size=min(k, len(swap_candidates)),
    replace=False,
    p=probabilities,
)
print(sampled_swaps)
print(probabilities)

[3 2]
[0.09444551 0.05514076 0.27042723 0.5799865 ]


In [18]:
from qiskit_gym.rl import RLSynthesis
from twisterl.utils import pull_hub_algorithm
import numpy as np

local_path = pull_hub_algorithm(
    repo_id="Qiskit/ai-transpiler_permutations",
    model_path="./models",
    revision="main",
    validate=True,
)

if not local_path:
    raise ValueError("Failed to download model from hub")

num_qubits = 12
seed = 42
input_perm = np.random.default_rng(seed).permutation(num_qubits).tolist()
input_perm = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0]


print(input_perm)

rls = RLSynthesis.from_config_json(
    f"{local_path}/permutation_12qO.json",
    f"{local_path}/permutation_12qO.safetensors",
)
qc_perm_output = rls.synth(
    input_perm, num_searches=10, num_mcts_searches=0, deterministic=False
)
print(qc_perm_output)

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 27962.03it/s]
2026-02-20 12:48:25.523 | INFO     | twisterl.utils:pull_hub_algorithm:124 - Model files are now in: ./models/models--Qiskit--ai-transpiler_permutations/snapshots/093574bc6cd8e39afa0c16a59318800b2e33b66a


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [NaN, NaN, 0.0, 0.0, NaN, 0.0, 0.0, 0.0, NaN, NaN, NaN, NaN]
Failed to create WeightedIndex: AllWeightsZero
The problematic probs were: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [NaN, NaN, 0.0, 0.0, NaN, 0.0, 0.0, 0.0, NaN, NaN, NaN, NaN]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [NaN, NaN, 0.0, 0.0, NaN, 0.0, 0.0, 0.0, NaN, NaN, NaN, NaN]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [0.0, NaN, 0.0, 0.0, NaN, 0.0, NaN, 0.0, NaN, NaN, NaN, 0.0]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN, NaN]
Failed to create WeightedIndex: InvalidWeight
The problematic probs were: [NaN, NaN, 0.0, 0.0, NaN, 0.0, 0.0, 0.0, NaN, NaN, NaN, NaN]
Failed to creat

In [3]:
from qiskit_gym.rl import RLSynthesis
from twisterl.utils import pull_hub_algorithm
from qiskit.quantum_info import random_clifford

repo_id = "Qiskit/ai-transpiler_cliffords"  # adjust if your repo name differs
model_stem = "clifford_9qL"  # replace with the stem of your model files

local_path = pull_hub_algorithm(
    repo_id=repo_id,
    model_path="./models",
    revision="main",
    validate=True,
)
if not local_path:
    raise ValueError("Failed to download Clifford model from hub")

num_qubits = 9
seed = 42
target_clifford = random_clifford(num_qubits, seed=seed)

rls = RLSynthesis.from_config_json(
    f"{local_path}/{model_stem}.json",
    f"{local_path}/{model_stem}.safetensors",
)

qc_clifford = rls.synth(
    target_clifford,
    num_searches=10,
    num_mcts_searches=0,
    deterministic=False,
)
print(qc_clifford)

Fetching 55 files: 100%|██████████| 55/55 [00:00<00:00, 5076.62it/s]
2026-02-20 11:06:20.210 | INFO     | twisterl.utils:pull_hub_algorithm:124 - Model files are now in: ./models/models--Qiskit--ai-transpiler_cliffords/snapshots/b0c4cad6dd752476ac36fb910d0a19a533a16796


     ┌───┐┌───┐┌───┐┌───┐                                                  »
q_0: ┤ H ├┤ H ├┤ H ├┤ H ├────────────■─────────────────────────────────────»
     └───┘├───┤└───┘└───┘     ┌───┐┌─┴─┐┌───┐                    ┌───┐┌───┐»
q_1: ──■──┤ S ├────────────■──┤ X ├┤ X ├┤ S ├────────────────────┤ X ├┤ S ├»
     ┌─┴─┐├───┤┌───┐     ┌─┴─┐└─┬─┘├───┤└───┘┌───┐          ┌───┐└─┬─┘└───┘»
q_2: ┤ X ├┤ S ├┤ H ├──■──┤ X ├──■──┤ H ├──■──┤ S ├──────────┤ X ├──■───────»
     └───┘└───┘└───┘┌─┴─┐└───┘┌───┐└───┘┌─┴─┐└───┘     ┌───┐└─┬─┘          »
q_3: ───────────────┤ X ├──■──┤ S ├─────┤ X ├───────■──┤ H ├──■─────────■──»
     ┌───┐          └───┘┌─┴─┐├───┤┌───┐├───┤     ┌─┴─┐├───┤          ┌─┴─┐»
q_4: ┤ S ├───────────────┤ X ├┤ H ├┤ S ├┤ H ├──■──┤ X ├┤ H ├───────■──┤ X ├»
     ├───┤┌───┐┌───┐┌───┐├───┤├───┤└───┘├───┤┌─┴─┐└───┘└───┘     ┌─┴─┐├───┤»
q_5: ┤ S ├┤ S ├┤ S ├┤ S ├┤ X ├┤ S ├─────┤ X ├┤ X ├────────────■──┤ X ├┤ S ├»
     └───┘├───┤├───┤├───┤└─┬─┘├───┤┌───┐└─┬─┘├───┤     ┌───┐┌─┴─┐├───┤├───┤»

In [2]:
model = rls.init_algorithm()

In [22]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

repo_id = "Qiskit/ai-transpiler_permutations"
filename = "permutation_12qO.safetensors"

path = hf_hub_download(repo_id=repo_id, filename=filename)
state = load_file(path)

# Print all parameter tensors and their shapes
for k, v in state.items():
    print(k)

action.0.bias
action.0.weight
common.0.bias
common.0.weight
embeddings.bias
embeddings.weight
value.0.bias
value.0.weight
